# PROJETO: Recomendação de Produtos via Classificação e Probabilidade de Compra
## Descrição
- Este script realiza a leitura dos arquivos da base de dados (clientes, vendas,
  crediario, promocoes, estoque), constrói um dataset consolidado de atributos (X)
  e variável alvo (Y), treina modelos de classificação da biblioteca scikit-learn
  para prever a categoria/produto de maior propensão de compra e gera gráficos
  personalizados com matplotlib e seaborn.

In [29]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [30]:
warnings.filterwarnings('ignore')

In [31]:
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

In [52]:
# =============================================================================
# 1. CARGA E LIMPEZA INICIAL DOS DADOS (CORRIGIDO)
# =============================================================================
def carregar_dados():
    """Carrega todos os arquivos CSV do projeto e força a conversão dos tipos de dados."""
    print("[1/6] Carregando bases de dados...")

    df_clientes = pd.read_csv('../data/clientes.csv')
    df_vendas = pd.read_csv('../data/vendas.csv')
    df_crediario = pd.read_csv('../data/crediario.csv')
    df_promocoes = pd.read_csv('../data/promocoes.csv')
    df_estoque = pd.read_csv('../data/estoque.csv')

    # Removendo duplicatas de cadastro de clientes
    df_clientes = df_clientes.drop_duplicates(subset=['id_cliente']).copy()

    # Garantir conversão estrita de colunas numéricas em vendas
    df_vendas['valor_total'] = pd.to_numeric(
        df_vendas['valor_total'].astype(str).str.replace(',', '.'), errors='coerce'
    ).fillna(0.0)

    df_vendas['desconto'] = pd.to_numeric(
        df_vendas['desconto'].astype(str).str.replace(',', '.'), errors='coerce'
    ).fillna(0.0)

    df_vendas['quantidade'] = pd.to_numeric(
        df_vendas['quantidade'], errors='coerce'
    ).fillna(1)

    df_vendas['data_venda'] = pd.to_datetime(df_vendas['data_venda'], errors='coerce')
    df_clientes['data_cadastro'] = pd.to_datetime(df_clientes['data_cadastro'], errors='coerce')

    return df_clientes, df_vendas, df_crediario, df_promocoes, df_estoque

In [45]:
# =============================================================================
# 2. ENGENHARIA DE ATRIBUTOS AVANÇADA (CORRIGIDO)
# =============================================================================
def criar_atributos(df_clientes, df_vendas, df_crediario, df_promocoes, df_estoque):
    """
    Consolida as features (X) por cliente com atributos de recorrência,
    proporção histórica por categoria e métricas financeiras comparativas.
    """
    print("[2/6] Realizando Engenharia de Atributos Avançada...")

    data_referencia = pd.to_datetime('2026-10-13')
    vendas_validas = df_vendas.dropna(subset=['id_cliente']).copy()
    vendas_validas['id_cliente'] = vendas_validas['id_cliente'].astype(int)

    # Conversão defensiva caso alguma coluna tenha mantido formato texto
    vendas_validas['valor_total'] = pd.to_numeric(vendas_validas['valor_total'], errors='coerce').fillna(0.0)
    vendas_validas['desconto'] = pd.to_numeric(vendas_validas['desconto'], errors='coerce').fillna(0.0)
    vendas_validas['quantidade'] = pd.to_numeric(vendas_validas['quantidade'], errors='coerce').fillna(1)

    vendas_validas['teve_desconto'] = (vendas_validas['desconto'] > 0).astype(int)

    # --- A. Preço médio global por categoria ---
    preco_medio_geral = vendas_validas['valor_total'].mean()

    # --- B. Agregações por Cliente ---
    agg_vendas = vendas_validas.groupby('id_cliente').agg(
        total_compras=('id_venda', 'nunique'),
        total_gasto=('valor_total', 'sum'),
        ticket_medio=('valor_total', 'mean'),
        desconto_medio=('desconto', 'mean'),
        proporcao_promocao=('teve_desconto', 'mean'),
        qtd_itens_total=('quantidade', 'sum'),
        ultima_compra=('data_venda', 'max'),
        forma_pagto_preferida=('forma_pagamento', lambda x: x.mode()[0] if not x.empty else 'Não informado')
    ).reset_index()

    # Cálculo dos dias desde a última compra
    if 'ultima_compra' in agg_vendas.columns:
        agg_vendas['dias_desde_ultima_compra'] = (
            data_referencia - pd.to_datetime(agg_vendas['ultima_compra'], errors='coerce')
        ).dt.days.fillna(999)
        agg_vendas.drop(columns=['ultima_compra'], inplace=True)
    else:
        agg_vendas['dias_desde_ultima_compra'] = 999

    # Feature: Ticket médio do cliente comparado ao preço médio geral
    agg_vendas['razao_ticket_preco_medio'] = agg_vendas['ticket_medio'] / (preco_medio_geral + 1e-5)

    # --- C. % de compras por categoria histórica ---
    cat_counts = vendas_validas.groupby(['id_cliente', 'categoria']).size().unstack(fill_value=0)
    cat_proporcoes = cat_counts.div(cat_counts.sum(axis=1), axis=0)
    cat_proporcoes.columns = [f'pct_compras_{col.lower().replace(" ", "_")}' for col in cat_proporcoes.columns]
    cat_proporcoes = cat_proporcoes.reset_index()

    # Variável Alvo (Y): Categoria principal de maior frequência
    categoria_preferida = cat_counts.idxmax(axis=1).reset_index()
    categoria_preferida.columns = ['id_cliente', 'categoria_alvo']

    # --- D. Atributos Financeiros e de Crédito ---
    agg_crediario = df_crediario.groupby('id_cliente').agg(
        total_parcelas=('id_parcela', 'count'),
        dias_atraso_medio=('dias_atraso', 'mean'),
        dias_atraso_maximo=('dias_atraso', 'max'),
        valor_em_aberto_total=('valor_em_aberto', 'sum'),
        taxa_inadimplencia=('status', lambda x: np.mean(x == 'Em atraso') if len(x) > 0 else 0.0)
    ).reset_index()

    # --- E. Junção dos DataFrames ---
    df_ml = df_clientes.merge(agg_vendas, on='id_cliente', how='left')
    df_ml = df_ml.merge(cat_proporcoes, on='id_cliente', how='left')
    df_ml = df_ml.merge(agg_crediario, on='id_cliente', how='left')
    df_ml = df_ml.merge(categoria_preferida, on='id_cliente', how='left')

    # Preenchimento de Nulos para Clientes Sem Histórico
    df_ml['total_compras'] = df_ml['total_compras'].fillna(0)
    df_ml['total_gasto'] = df_ml['total_gasto'].fillna(0.0)
    df_ml['ticket_medio'] = df_ml['ticket_medio'].fillna(0.0)
    df_ml['desconto_medio'] = df_ml['desconto_medio'].fillna(0.0)
    df_ml['proporcao_promocao'] = df_ml['proporcao_promocao'].fillna(0.0)
    df_ml['qtd_itens_total'] = df_ml['qtd_itens_total'].fillna(0)
    df_ml['dias_desde_ultima_compra'] = df_ml['dias_desde_ultima_compra'].fillna(999)
    df_ml['razao_ticket_preco_medio'] = df_ml['razao_ticket_preco_medio'].fillna(0.0)
    df_ml['forma_pagto_preferida'] = df_ml['forma_pagto_preferida'].fillna('Sem Histórico')

    # Preenche colunas de proporção por categoria com 0.0
    pct_cols = [c for c in df_ml.columns if c.startswith('pct_compras_')]
    df_ml[pct_cols] = df_ml[pct_cols].fillna(0.0)

    # Preenche colunas do crediário
    df_ml['total_parcelas'] = df_ml['total_parcelas'].fillna(0)
    df_ml['dias_atraso_medio'] = df_ml['dias_atraso_medio'].fillna(0)
    df_ml['dias_atraso_maximo'] = df_ml['dias_atraso_maximo'].fillna(0)
    df_ml['valor_em_aberto_total'] = df_ml['valor_em_aberto_total'].fillna(0.0)
    df_ml['taxa_inadimplencia'] = df_ml['taxa_inadimplencia'].fillna(0.0)

    # Cálculo do tempo de cadastro em dias
    df_ml['dias_cadastro'] = (data_referencia - pd.to_datetime(df_ml['data_cadastro'], errors='coerce')).dt.days.fillna(0)

    return df_ml, pct_cols

In [34]:
# =============================================================================
# 3. PIPELINE DE PRÉ-PROCESSAMENTO
# =============================================================================
def preparar_pipeline(X_train):
    """Cria os pipelines de transformação numérica e categórica."""
    print("[3/6] Configurando pipelines de transformação...")

    num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, num_cols),
            ('cat', cat_transformer, cat_cols)
        ]
    )

    return preprocessor, num_cols, cat_cols

In [35]:
# =============================================================================
# 4. TREINAMENTO COM BALANCEAMENTO DE CLASSES E MODELAGEM
# =============================================================================
def treinar_e_avaliar_modelos(df_ml, pct_cols):
    """Divide os dados, treina classificadores balanceados e avalia os resultados."""
    print("[4/6] Treinando classificadores com balanceamento de classes...")

    # Apenas clientes com histórico de compra entram no treino do modelo supervisionado
    df_treino_val = df_ml.dropna(subset=['categoria_alvo']).copy()

    feature_cols = [
        'idade', 'cidade', 'canal_preferencial', 'total_compras', 'total_gasto',
        'ticket_medio', 'desconto_medio', 'proporcao_promocao', 'qtd_itens_total',
        'forma_pagto_preferida', 'dias_desde_ultima_compra', 'razao_ticket_preco_medio',
        'total_parcelas', 'dias_atraso_medio', 'dias_atraso_maximo',
        'valor_em_aberto_total', 'taxa_inadimplencia', 'dias_cadastro'
    ] + pct_cols

    X = df_treino_val[feature_cols]
    y = df_treino_val['categoria_alvo']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    preprocessor, num_cols, cat_cols = preparar_pipeline(X_train)

    # Modelos com pesagem balanceada para lidar com desequilíbrio de categorias
    modelos = {
        'Random Forest (Balanced)': RandomForestClassifier(
            n_estimators=200, max_depth=10, class_weight='balanced', random_state=42
        ),
        'Regressão Logística (Balanced)': LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=42
        ),
        'Gradient Boosting (OneVsRest)': OneVsRestClassifier(
            GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
        )
    }

    resultados = {}
    melhor_modelo = None
    melhor_f1 = -1.0
    melhor_nome = ""

    for nome, clf in modelos.items():
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', clf)
        ])

        cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1_weighted')
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')

        resultados[nome] = {
            'Pipeline': pipeline,
            'Accuracy': acc,
            'F1-Score': f1,
            'CV F1 Mean': cv_scores.mean(),
            'y_pred': y_pred
        }

        print(f"   -> {nome}: Acurácia = {acc:.4f} | F1-Score Weighted = {f1:.4f} | Cross-Val F1 = {cv_scores.mean():.4f}")

        if f1 > melhor_f1:
            melhor_f1 = f1
            melhor_modelo = pipeline
            melhor_nome = nome

    print(f"\n=> Melhor modelo selecionado: {melhor_nome} (F1-Score: {melhor_f1:.4f})")
    return melhor_modelo, melhor_nome, X_train, X_test, y_train, y_test, resultados, num_cols, cat_cols, feature_cols

In [36]:
# =============================================================================
# 5. GERADOR DE VISUALIZAÇÕES E AVALIAÇÃO
# =============================================================================
def gerar_visualizacoes(melhor_modelo, melhor_nome, X_test, y_test, y_pred, num_cols, cat_cols):
    """Gera Matriz de Confusão e Importância de Atributos."""
    print("[5/6] Gerando gráficos de desempenho...")

    # Matriz de Confusão
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_test, y_pred)
    labels = np.unique(y_test)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(f'Matriz de Confusão - {melhor_nome}', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Categoria Predita (Recomendada)', fontsize=11)
    plt.ylabel('Categoria Real Adquirida', fontsize=11)
    plt.tight_layout()
    plt.savefig('matriz_confusao.png', dpi=300)
    plt.close()
    print("   -> Gráfico salvo: matriz_confusao.png")

    # Importância dos Atributos
    classifier = melhor_modelo.named_steps['classifier']
    if hasattr(classifier, 'feature_importances_'):
        ohe = melhor_modelo.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
        cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist()
        all_feature_names = num_cols + cat_feature_names

        importances = classifier.feature_importances_
        df_imp = pd.DataFrame({'Atributo': all_feature_names, 'Importância': importances})
        df_imp = df_imp.sort_values(by='Importância', ascending=False).head(12)

        plt.figure(figsize=(10, 6))
        sns.barplot(data=df_imp, x='Importância', y='Atributo', palette='viridis')
        plt.title('Top 12 Atributos Relevantes na Recomendação', fontsize=13, fontweight='bold', pad=15)
        plt.xlabel('Grau de Importância Relativa', fontsize=11)
        plt.ylabel('Atributo', fontsize=11)
        plt.tight_layout()
        plt.savefig('importancia_atributos.png', dpi=300)
        plt.close()
        print("   -> Gráfico salvo: importancia_atributos.png")

In [49]:
# =============================================================================
# 6. SISTEMA HÍBRIDO DE RECOMENDAÇÃO (COM FALLBACK E DETECÇÃO DINÂMICA)
# =============================================================================
def obter_categorias_fallback(df_vendas, df_estoque, df_promocoes, top_n=3):
    """
    Regra de Fallback para clientes novos (sem histórico):
    Combina popularidade nas vendas, disponibilidade em estoque e descontos ativos nas promoções.
    """
    # 1. Volume de Vendas por Categoria
    if 'categoria' in df_vendas.columns:
        pop = df_vendas.groupby('categoria').size().rename('qtd_vendas')
    else:
        pop = pd.Series(dtype=float)

    # 2. Busca dinâmica da coluna de desconto em df_promocoes
    cols_desconto_possiveis = ['desconto_percentual', 'desconto', 'percentual_desconto', 'val_desconto', 'pct_desconto']
    cols_encontradas = [c for c in cols_desconto_possiveis if c in df_promocoes.columns]

    if 'categoria' in df_promocoes.columns and len(cols_encontradas) > 0:
        col_desc = cols_encontradas[0]
        df_promos_temp = df_promocoes.copy()
        df_promos_temp[col_desc] = pd.to_numeric(df_promos_temp[col_desc], errors='coerce').fillna(0.0)
        promos = df_promos_temp.groupby('categoria')[col_desc].max().rename('maior_desconto')
    else:
        promos = pd.Series(dtype=float)

    # 3. Consolidação e Cálculo do Score de Popularidade
    df_pop = pd.DataFrame(pop).join(promos, how='left').fillna(0.0)

    if 'maior_desconto' not in df_pop.columns:
        df_pop['maior_desconto'] = 0.0

    vendas_max = df_pop['qtd_vendas'].max() if not df_pop.empty and df_pop['qtd_vendas'].max() > 0 else 1.0

    # Score Híbrido: 70% peso do volume de vendas + 30% bônus de desconto ativo
    df_pop['score_fallback'] = (df_pop['qtd_vendas'] / vendas_max) * 70.0 + df_pop['maior_desconto'] * 0.3

    ranking_fallback = df_pop.sort_values(by='score_fallback', ascending=False).head(top_n).reset_index()
    ranking_fallback = ranking_fallback.rename(
        columns={'categoria': 'Categoria Recomendada', 'score_fallback': 'Score de Popularidade/Promoção (%)'}
    )

    return ranking_fallback[['Categoria Recomendada', 'Score de Popularidade/Promoção (%)']]

In [53]:
# =============================================================================
# FLUXO PRINCIPAL DE EXECUÇÃO
# =============================================================================
if __name__ == "__main__":
    print("=" * 70)
    print(" PIPELINE DE MACHINE LEARNING COM ENGENHARIA DE RECORRÊNCIA E FALLBACK")
    print("=" * 70)

    # 1. Carregar dados
    df_clientes, df_vendas, df_crediario, df_promocoes, df_estoque = carregar_dados()

    # 2. Engenharia de Recursos Avançada
    df_ml, pct_cols = criar_atributos(df_clientes, df_vendas, df_crediario, df_promocoes, df_estoque)

    # 3. Treinar e Avaliar
    melhor_modelo, melhor_nome, X_train, X_test, y_train, y_test, resultados, num_cols, cat_cols, feature_cols = treinar_e_avaliar_modelos(df_ml, pct_cols)

    # 4. Avaliação Detalhada no Teste
    y_pred = resultados[melhor_nome]['y_pred']
    print("\n" + "=" * 70)
    print(f" RELATÓRIO DETALHADO DE DESEMPENHO ({melhor_nome.upper()})")
    print("=" * 70)
    print(classification_report(y_test, y_pred))

    # 5. Gerar Gráficos
    gerar_visualizacoes(melhor_modelo, melhor_nome, X_test, y_test, y_pred, num_cols, cat_cols)

    # 6. Demonstração de Recomendação Híbrida
    print("\n" + "=" * 70)
    print(" DEMONSTRAÇÃO PRÁTICA: RECOMENDAÇÃO PERSONALIZADA")
    print("=" * 70)

    # Exemplo 1: Cliente com histórico
    id_com_historico = df_ml[df_ml['total_compras'] > 0]['id_cliente'].iloc[0]
    print(f"\n--- Recomendação para Cliente com Histórico (ID = {id_com_historico}) ---")
    rec1 = recomendar_para_cliente_hibrido(
        melhor_modelo, df_ml, id_com_historico, df_vendas, df_estoque, df_promocoes, feature_cols, top_n=3
    )
    print(rec1.to_string(index=False))

    # Exemplo 2: Cliente sem histórico (Cold Start)
    id_sem_historico = df_ml[df_ml['total_compras'] == 0]['id_cliente'].iloc[0] if (df_ml['total_compras'] == 0).any() else 99999
    print(f"\n--- Recomendação para Cliente Novo / Cold Start (ID = {id_sem_historico}) ---")
    rec2 = recomendar_para_cliente_hibrido(
        melhor_modelo, df_ml, id_sem_historico, df_vendas, df_estoque, df_promocoes, feature_cols, top_n=3
    )
    print(rec2.to_string(index=False))

    print("=" * 70)
    print(" Pipeline concluído com sucesso!")

 PIPELINE DE MACHINE LEARNING COM ENGENHARIA DE RECORRÊNCIA E FALLBACK
[1/6] Carregando bases de dados...
[2/6] Realizando Engenharia de Atributos Avançada...
[4/6] Treinando classificadores com balanceamento de classes...
[3/6] Configurando pipelines de transformação...
   -> Random Forest (Balanced): Acurácia = 0.8667 | F1-Score Weighted = 0.8702 | Cross-Val F1 = 0.8717
   -> Regressão Logística (Balanced): Acurácia = 0.9200 | F1-Score Weighted = 0.9230 | Cross-Val F1 = 0.8832
   -> Gradient Boosting (OneVsRest): Acurácia = 0.8800 | F1-Score Weighted = 0.8775 | Cross-Val F1 = 0.8974

=> Melhor modelo selecionado: Regressão Logística (Balanced) (F1-Score: 0.9230)

 RELATÓRIO DETALHADO DE DESEMPENHO (REGRESSÃO LOGÍSTICA (BALANCED))
              precision    recall  f1-score   support

   Acessório       1.00      0.92      0.96        52
     Bermuda       1.00      0.81      0.90        32
       Blusa       0.62      1.00      0.77        10
       Calça       0.93      1.00      0.